# TC5061 | Actividad: Dataset - Notebook | Individual
**Maestría en Inteligencia Artificial Aplicada - Machine Learning Operations (MLOps)**
**Autor:** Daniel Sánchez Hernández

## Objetivo
Recorrer de forma reproducible el flujo básico de un proyecto de Machine Learning: explorar y limpiar un conjunto de datos, entrenar y evaluar un modelo base (*baseline*) y registrar el experimento en MLflow Tracking.

## Dataset
**AI4I 2020 Predictive Maintenance Dataset** (Matzka, S., 2020). Conjunto de datos sintético que simula datos de una fresadora industrial, pensado para tareas de mantenimiento predictivo.
- Fuente original: [Kaggle - stephanmatzka](https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020) y [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/601).
- Licencia: Creative Commons Attribution 4.0 (CC BY 4.0).
- Problema a resolver: **clasificación binaria** - predecir si la máquina falla (`Machine failure`) a partir de las condiciones de operación.

## Nota de reproducibilidad
Este notebook debe ejecutarse desde la raíz del repositorio, donde existe la carpeta `data/raw/`. La configuración global (rutas, semilla aleatoria y opciones de visualización) se define una sola vez, al inicio.

In [1]:
# Importación de librerías y registro de versiones (evidencia de reproducibilidad)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import seaborn as sns
import sklearn

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)
print("seaborn     :", sns.__version__)

Python      : 3.11.16
pandas      : 3.0.5
numpy       : 2.4.6
scikit-learn: 1.4.2
matplotlib  : 3.11.2
seaborn     : 0.13.2


## Configuración global

Todas las decisiones que aplican al notebook completo se concentran en esta celda, para no repetirlas en cada paso y para que cualquier persona sepa dónde cambiarlas:
- **Ruta de datos:** ubicación del archivo original, que se trata como solo lectura.
- **Semilla aleatoria (`SEED`):** se usará en la partición entrenamiento/prueba y en el modelo, para que los resultados sean reproducibles en cualquier ejecución.
- **Opciones de visualización:** pandas muestra todas las columnas y el texto completo de las celdas; las gráficas usan un estilo uniforme.

In [2]:
# Ruta del archivo original (solo lectura: nunca se modifica)
DATA_PATH = Path("data/raw/ai4i2020.csv")

# Semilla aleatoria única para todo el notebook (reproducibilidad)
SEED = 42

# Opciones de visualización de pandas
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
pd.set_option("display.width", 200)          # ancho suficiente para tablas amplias
pd.set_option("display.max_colwidth", None)  # no recortar textos largos

# Estilo uniforme para todas las gráficas
# sns.set_theme(style="whitegrid")

## Paso 1: Carga y descripción del dataset

En este paso se importan los datos y se describe su estructura: número de registros, variables, tipos de dato y significado de cada columna.

**Validación de esquema:** después de cargar el archivo se verifica que contenga exactamente las columnas esperadas. Es una práctica básica de validación de datos en MLOps: si la fuente cambia (una columna renombrada, faltante o adicional), el notebook se detiene con un mensaje claro en lugar de fallar más adelante de forma confusa. En pipelines productivos este control se automatiza con herramientas como Great Expectations o DeepChecks.

In [ ]:
# Carga del dataset
df = pd.read_csv(DATA_PATH)

# Esquema esperado: columnas documentadas por el autor del dataset
EXPECTED_COLUMNS = ["UDI", "Product ID", "Type", "Air temperature [K]", "Process temperature [K]",
                    "Rotational speed [rpm]", "Torque [Nm]", "Tool wear [min]",
                    "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF",
]

# Comparación entre columnas esperadas y columnas reales
missing_columns = set(EXPECTED_COLUMNS) - set(df.columns)
extra_columns = set(df.columns) - set(EXPECTED_COLUMNS)

assert not missing_columns, f"Faltan columnas: {missing_columns}"
assert not extra_columns, f"Columnas no esperadas: {extra_columns}"
print("Validación de esquema correcta: las", len(EXPECTED_COLUMNS), "columnas esperadas están presentes.")

Validación de esquema correcta: las 14 columnas esperadas están presentes.


In [5]:
# Dimensiones del dataset: (registros, variables)
n_rows, n_cols = df.shape
print(f"Registros: {n_rows}")
print(f"Variables: {n_cols}")

Registros: 10000
Variables: 14


In [6]:
# Primeras filas para ver el aspecto real de los datos
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [7]:
# Estructura: tipo de dato y valores no nulos por columna
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [8]:
# Cardinalidad: valores distintos por columna (ayuda a distinguir identificadores, categorías y numéricas)
df.nunique().to_frame(name="n_unique")

,n_unique
UDI,10000
Product ID,10000
Type,3
Air temperature [K],93
Process temperature [K],82
Rotational speed [rpm],941
Torque [Nm],577
Tool wear [min],246
Machine failure,2
TWF,2


### Diccionario de datos
La siguiente tabla documenta el rol, la descripción y la unidad de cada variable. Las descripciones se conservan en inglés, tal como aparecen en la documentación original del dataset (Matzka, 2020; [Kaggle](https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020); [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/601)). Se añade una columna con el tipo de dato que pandas asignó realmente, para contrastarlo con la documentación.

Observaciones:
- La documentación llama `UID` al identificador, pero en el archivo CSV la columna se llama `UDI`; se usa el nombre real del archivo.
- UCI clasifica a `TWF`, `HDF`, `PWF`, `OSF` y `RNF` como variables objetivo, no como predictores: son los cinco modos de falla y `Machine failure` vale 1 cuando ocurre al menos uno. Este punto se retoma al construir el modelo base (Paso 4).

In [9]:
# Diccionario de datos: descripciones tomadas de la documentación original (en inglés)
data_dictionary = pd.DataFrame(
    [
        ("UDI", "ID", "Unique identifier ranging from 1 to 10000", "-"),
        ("Product ID", "ID", "Consisting of a letter L, M, or H for low (50% of all products), medium (30%) and high (20%) as product quality variants and a variant-specific serial number", "-"),
        ("Type", "Feature", "Just the product type L, M or H from column 2", "-"),
        ("Air temperature [K]", "Feature", "Generated using a random walk process later normalized to a standard deviation of 2 K around 300 K", "K"),
        ("Process temperature [K]", "Feature", "Generated using a random walk process normalized to a standard deviation of 1 K, added to the air temperature plus 10 K", "K"),
        ("Rotational speed [rpm]", "Feature", "Calculated from a power of 2860 W, overlaid with a normally distributed noise", "rpm"),
        ("Torque [Nm]", "Feature", "Torque values are normally distributed around 40 Nm with a SD = 10 Nm and no negative values", "Nm"),
        ("Tool wear [min]", "Feature", "The quality variants H/M/L add 5/3/2 minutes of tool wear to the used tool in the process", "min"),
        ("Machine failure", "Target", "Label that indicates whether the machine has failed in this particular datapoint for any of the failure modes", "-"),
        ("TWF", "Target", "Tool wear failure", "-"),
        ("HDF", "Target", "Heat dissipation failure", "-"),
        ("PWF", "Target", "Power failure", "-"),
        ("OSF", "Target", "Overstrain failure", "-"),
        ("RNF", "Target", "Random failures", "-"),
    ],
    columns=["Variable", "Role", "Description", "Units"],
)

# Tipo de dato real asignado por pandas, para contrastarlo con la documentación
data_dictionary["pandas dtype"] = data_dictionary["Variable"].map(df.dtypes.astype(str))
data_dictionary

,Variable,Role,Description,Units,pandas dtype
0,UDI,ID,Unique identifier ranging from 1 to 10000,-,int64
1,Product ID,ID,"Consisting of a letter L, M, or H for low (50% of all products), medium (30%) and high (20%) as product quality variants and a variant-specific serial number",-,str
2,Type,Feature,"Just the product type L, M or H from column 2",-,str
3,Air temperature [K],Feature,Generated using a random walk process later normalized to a standard deviation of 2 K around 300 K,K,float64
4,Process temperature [K],Feature,"Generated using a random walk process normalized to a standard deviation of 1 K, added to the air temperature plus 10 K",K,float64
5,Rotational speed [rpm],Feature,"Calculated from a power of 2860 W, overlaid with a normally distributed noise",rpm,int64
6,Torque [Nm],Feature,Torque values are normally distributed around 40 Nm with a SD = 10 Nm and no negative values,Nm,float64
7,Tool wear [min],Feature,The quality variants H/M/L add 5/3/2 minutes of tool wear to the used tool in the process,min,int64
8,Machine failure,Target,Label that indicates whether the machine has failed in this particular datapoint for any of the failure modes,-,int64
9,TWF,Target,Tool wear failure,-,int64
